# p2 db schematics

## Goal
stand up Postgres with pgvector, design schema, set up Alembic migrations, load p1 metadata with empty embeddings.

1. spin up psql + pgvector (local docker first, terraform later!)
2. Table setup, baseline first, userdefined later.
3. Vector search setup -HNSW index on embedding column
4. alembic now or later?
5. store metadata from p1

In [ ]:
CREATE TABLE patients (
    patient_id       TEXT PRIMARY KEY,
    sex              TEXT,
    birth_date       DATE
);

CREATE TABLE studies (
    study_instance_uid  TEXT PRIMARY KEY,
    patient_id          TEXT REFERENCES patients(patient_id),
    study_date          DATE,
    study_description   TEXT,      -- e.g. 'normal_07'
    accession_number     TEXT
);

CREATE TABLE images (
    sop_instance_uid     TEXT PRIMARY KEY,
    series_instance_uid  TEXT NOT NULL,
    study_instance_uid   TEXT REFERENCES studies(study_instance_uid),
    patient_id           TEXT REFERENCES patients(patient_id), 
    laterality            TEXT CHECK (laterality IN ('L','R')),
    view                  TEXT CHECK (view IN ('CC','MLO')),
    file_path             TEXT NOT NULL,
    rows                   INT,
    columns                INT,
    pixel_spacing_mm       FLOAT8,
    bits_stored             INT,
    window_center           FLOAT8,
    window_width             FLOAT8,
    photometric_interpretation TEXT,
    embedding                VECTOR(768),   -- dim depends on chosen encoder
    label                     TEXT,          -- 'normal' / 'abnormal' / or whatever the user inputs?
    created_at                TIMESTAMPTZ DEFAULT now()
);

CREATE INDEX idx_images_embedding ON images
    USING hnsw (embedding vector_cosine_ops);

CREATE INDEX idx_images_patient_study ON images (patient_id, study_instance_uid);